# GRACE H3 — Bắt đầu full pipeline tại đây

Hướng mới: **CPU logistic gate → Platt calibration → constrained routing → GRACE cho inspect**. Mặc định full test Devign, skip-low; direct-high tắt. Gate-only và full GRACE là đối chứng. Không có GPU/Kaggle stage ở cấu hình H3 độc lập.

| Bước | Môi trường | Handoff |
|---|---|---|
| 1 | Local | Xuất cpu-inputs.zip; chuyển sang colab_cpu_prepare.ipynb |
| 2 | Colab CPU, accelerator None | Fit gate/calibration/thresholds, route, prepare GRACE; trả cpu-output.zip |
| 3 | Local | Import, preflight → Vertex → merge full cohort → paired evaluation |

Notebook dừng ở Bước1 khi thiếu CPU output. Tải output vào bundles/ rồi Run All lại để tiếp tục. full_reference=true thêm các query bị skip vào nhánh đối chứng; cascade chạy inference trước để số request của nó không bị che bởi prewarming cache.

Đọc docs/METHOD.md về ràng buộc lỗi, docs/RUNBOOK.md về cloud và H1+H3 tùy chọn.

In [4]:
from pathlib import Path
import os,sys,json,subprocess,zipfile
candidates=([Path(os.environ['GRACE_H3_ROOT'])] if os.environ.get('GRACE_H3_ROOT') else [])
candidates += [Path.cwd(),*Path.cwd().parents,Path('D:/JUNIOR/LabThayTho/GRACE-H3-CalibratedCascade'),Path('/mnt/d/JUNIOR/LabThayTho/GRACE-H3-CalibratedCascade')]
ROOT=next((p.resolve() for p in candidates if (p/'grace_h3').is_dir()),None)
if ROOT is None: raise RuntimeError('Set GRACE_H3_ROOT to this project root')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
INSTALL_DEPENDENCIES=True
PREPARE_ON_COLAB=True
RUN_INFERENCE=True
OUTPUT_BUNDLE=ROOT/'bundles'/'grace-h3-cpu-output.zip'
H1_HEADS_BUNDLE=ROOT/'bundles'/'grace-h1-trained-heads.zip' # optional H1+H3 only
config=json.loads((ROOT/'configs'/'experiment.json').read_text(encoding='utf-8'))
print(ROOT);print(json.dumps(config,indent=2))

/mnt/d/JUNIOR/LabThayTho/GRACE-H3-CalibratedCascade
{
  "datasets": [
    "devign"
  ],
  "split": "test",
  "full_reference": true,
  "retrieval_arm": "baseline_whitening",
  "graph_workers": 2,
  "inference_workers": 4,
  "bootstrap": 1000,
  "gate": {
    "C": 1,
    "class_weight": null,
    "max_iter": 1000,
    "seed": 42,
    "calibration_fraction": 0.5,
    "low_grid": [
      0.01,
      0.025,
      0.05,
      0.075,
      0.1,
      0.15,
      0.2,
      0.25,
      0.3
    ],
    "high_grid": [
      0.7,
      0.8,
      0.9,
      0.95,
      0.975,
      0.99
    ],
    "enable_direct_high": false,
    "max_lost_positive_fraction": 0.01,
    "min_high_precision": 0.98,
    "max_high_false_positive_fraction": 0.01,
    "family_alpha": 0.05
  }
}


In [5]:
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements.txt')])
from grace_h1.io import read_json,file_hash,load_local_environment
from grace_h3.portable import create_bundle,import_bundle,identity
from grace_h3.workflow import prepare_all,selected_jobs,head_for,preflight_all,infer_all,evaluate_all
import pandas as pd
load_local_environment(ROOT)

## Bước 1 — Chuyển phần CPU/RAM sang Colab

Lần đầu: mang grace-h3-cpu-inputs.zip và grace-h3-joern-runtime.zip trong bundles/ lên MyDrive/GRACE-H3-transfer. Upload notebooks/colab_cpu_prepare.ipynb vào Colab, chọn accelerator **None**, Run All.

Colab dùng train để fit logistic, chia validation thành calibration và threshold selection riêng, khóa policy, rồi chỉ prepare GRACE cho inspect. Khi full_reference=true, Colab prepare thêm full cohort cho đối chứng. Không gọi LLM trên Colab.

Tải grace-h3-cpu-output.zip về bundles/ local, Run All notebook này lại. Có thể chọn PREPARE_ON_COLAB=False để thực hiện offline trên local với Joern.

**H1+H3 tùy chọn:** sửa retrieval_arm trong configs/experiment.json thành h1_supcon_hard trước khi xuất bundle; mang trained-heads.zip từ project H1 về H1_HEADS_BUNDLE. Nếu chưa train H1, chạy notebook Kaggle của project H1 trước. Hai nhánh reference/cascade đều dùng cùng H1 head; gate H3 vẫn dùng frozen CodeT5.

In [6]:
if config['retrieval_arm']!='baseline_whitening':
    if H1_HEADS_BUNDLE.exists(): print(import_bundle(ROOT,H1_HEADS_BUNDLE,'trained_heads'))
    for dataset in config['datasets']:
        try: head_for(ROOT,dataset,config['retrieval_arm'])
        except (FileNotFoundError,KeyError,ValueError) as exc:
            raise RuntimeError('H1+H3 cần trained-heads.zip matching từ project H1; đặt tại H1_HEADS_BUNDLE. H3 độc lập dùng retrieval_arm=baseline_whitening.') from exc
if PREPARE_ON_COLAB:
    if OUTPUT_BUNDLE.exists(): print(import_bundle(ROOT,OUTPUT_BUNDLE,'cpu_output'))
    try:
        jobs=selected_jobs(ROOT,config)
    except (FileNotFoundError,KeyError,ValueError) as exc:
        bundle=ROOT/'bundles'/'grace-h3-cpu-inputs.zip';ready=False
        if bundle.exists() and config['retrieval_arm']=='baseline_whitening':
            with zipfile.ZipFile(bundle) as z:
                m=json.loads(z.read('bundle_manifest.json'))
                ready=m.get('schema')=='grace_h3_bundle_v1' and m.get('kind')=='cpu_inputs' and m['job']=={'source':identity(ROOT),'data_manifest':file_hash(ROOT/'data'/'manifest.json'),'config':config}
        print({'reuse_existing_inputs':ready,'path':str(bundle)} if ready else create_bundle(ROOT,bundle,'cpu_inputs'))
        raise RuntimeError('DỪNG: chạy notebooks/colab_cpu_prepare.ipynb trên Colab CPU với cpu-inputs.zip và joern-runtime.zip; tải cpu-output.zip vào bundles/ rồi Run All lại. '+str(exc)) from exc
else:
    prepare_all(ROOT,config);jobs=selected_jobs(ROOT,config)
print('Prepared jobs:',[j['dataset'] for _,j in jobs])

{'kind': 'cpu_output', 'copied': 272, 'reused_identical': 0}
Prepared jobs: ['devign']


## Bước 2 — Đọc policy đã khóa

low=0 nghĩa là chưa có ngưỡng skip đạt constraint; tất cả đi inspect. high=null nghĩa là direct-high chưa bật/không đạt constraint. Bound Clopper–Pearson/Bonferroni chỉ có ý nghĩa nominal trên assumptions đã ghi; không bảo đảm recall test hoặc domain mới. Không đổi thresholds sau khi xem test.

In [7]:
policies=[]
for folder,job in jobs:
    g=read_json(folder/'gate_manifest.json')
    policies.append({'dataset':job['dataset'],'low':g['policy']['low'],'high':g['policy']['high'],
                     'branches':job['branch_counts'],'selection_n':g['counts']['selection'],
                     'selection_brier':g['selection_calibrated']['brier'],'fit_seconds':g['fit_seconds']})
display(pd.DataFrame(policies))

,dataset,low,high,branches,selection_n,selection_brier,fit_seconds
0,devign,0.075,None,"{'inspect': 2726, 'skip_low': 6}",1362,0.223904,34.780694


## Bước 3 — Preflight rồi inference local

RUN_INFERENCE=True gọi Vertex cho exact prompts còn thiếu. Cascade gọi trước; full reference sau có thể dùng lại response inspect. full_reference=true có thể khiến cả benchmark vẫn gọi LLM cho toàn bộ cohort, dù deployed cascade chỉ cần tỷ lệ inspect. Báo cáo tách chi phí reference, routing và cache.

ADC/Vertex dùng môi trường local; không đưa credential lên Colab. Parser invalid không bị ép thành safe.

In [8]:
plans=preflight_all(ROOT,config)
for p in plans: print(json.dumps(p,indent=2))

{
  "dataset": "devign",
  "n": 2732,
  "branches": {
    "inspect": 2726,
    "skip_low": 6
  },
  "algorithmic_llm_query_ratio": 0.9978038067349927,
  "cascade": {
    "queries": 2726,
    "unique_prompts": 2725,
    "cached_queries": 2159,
    "new_unique_requests": 566,
    "imported_prior_responses": 2731,
    "request_spec": {
      "model": "publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas",
      "provider": "Vertex AI",
      "inference": {
        "temperature": 0.0,
        "seed": 42,
        "max_tokens": 256,
        "retry_policy": "max_attempts=5; exponential_backoff=5s..120s; exhausted_request=invalid",
        "parser": "../Grace-with-filters-v2/[CONTRACT]/parser/parse_prediction.py::parse_prediction",
        "parser_contract": "exact_last_line_binary_v1"
      }
    },
    "worst_case_attempts": 2830
  },
  "reference": {
    "queries": 2732,
    "unique_prompts": 2731,
    "cached_queries": 2162,
    "new_unique_requests": 569,
    "imported_prior_respons

In [9]:
reports=infer_all(ROOT,config,allow_api=RUN_INFERENCE)
for report in reports: print(json.dumps(report,indent=2))

LLM unique requests 25/566
LLM unique requests 50/566
LLM unique requests 75/566
LLM unique requests 100/566
LLM unique requests 125/566
LLM unique requests 150/566
LLM unique requests 175/566
LLM unique requests 200/566
LLM unique requests 225/566
LLM unique requests 250/566
LLM unique requests 275/566
LLM unique requests 300/566
LLM unique requests 325/566
LLM unique requests 350/566
LLM unique requests 375/566
LLM unique requests 400/566
LLM unique requests 425/566
LLM unique requests 450/566
LLM unique requests 475/566
LLM unique requests 500/566
LLM unique requests 525/566
LLM unique requests 550/566
LLM unique requests 566/566
LLM unique requests 3/3
{
  "dataset": "devign",
  "cascade": {
    "queries": 2726,
    "unique_prompts": 2725,
    "cached_queries": 2159,
    "new_unique_requests": 566,
    "imported_prior_responses": 0,
    "request_spec": {
      "model": "publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas",
      "provider": "Vertex AI",
      "inference": {


## Bước 4 — Đánh giá toàn cohort

Có cascade, gate-only, full reference, branch FN/FP, invalid rate, recall_all_positives và paired CI. ROC-AUC/AP chỉ thuộc gate score, không gắn tên AUC của final cascade. Các metric F1/MCC condition trên valid; invalid được giữ trong báo cáo full cohort.

In [10]:
summary=evaluate_all(ROOT,config)
rows=[]
for r in summary['datasets']:
    for arm in ['cascade','gate_only','full_reference']:
        if arm in r:
            m=r[arm];rows.append({'dataset':r['dataset'],'arm':arm,'n':m['n'],'f1_valid':m['f1'],'mcc_valid':m['mcc'],
                                 'invalid_rate':m['invalid_rate'],'recall_all_positives':m['recall_all_positives']})
    print('Branches:',r['branches'],'algorithmic saving:',r['algorithmic_call_saving'])
    if 'paired_reference_to_cascade' in r: print(json.dumps(r['paired_reference_to_cascade'],indent=2))
display(pd.DataFrame(rows))
print('Status:',summary['status'],'saved:',ROOT/'runs'/'experiment_summary.json')

Branches: {'skip_low': {'n': 6, 'coverage': 0.0021961932650073207, 'metrics': {'n': 6, 'valid_n': 6, 'invalid_n': 0, 'invalid_rate': 0.0, 'confusion_matrix_valid_only': {'tp': 0, 'tn': 6, 'fp': 0, 'fn': 0}, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'mcc': 0.0, 'balanced_accuracy': 0.5, 'accuracy_valid_only': 1.0, 'accuracy_all': 1.0, 'recall_all_positives': 0.0, 'predicted_positive_rate_all': 0.0, 'invalid_positive_n': 0, 'metric_scope': 'binary metrics condition on valid labels; invalids retained and reported separately'}, 'skip_fn': 0, 'high_fp': None, 'skip_lost_positive_fraction': 0.0}, 'direct_high': {'n': 0, 'coverage': 0.0, 'metrics': None, 'skip_fn': None, 'high_fp': 0, 'skip_lost_positive_fraction': None}, 'inspect': {'n': 2726, 'coverage': 0.9978038067349927, 'metrics': {'n': 2726, 'valid_n': 2726, 'invalid_n': 0, 'invalid_rate': 0.0, 'confusion_matrix_valid_only': {'tp': 339, 'tn': 1141, 'fp': 330, 'fn': 916}, 'precision': 0.5067264573991032, 'recall': 0.2701195219123506, 

,dataset,arm,n,f1_valid,mcc_valid,invalid_rate,recall_all_positives
0,devign,cascade,2732,0.352391,0.054114,0.0,0.270120
1,devign,gate_only,2732,0.510261,0.217310,0.0,0.435857
2,devign,full_reference,2732,0.352208,0.053302,0.0,0.270120


Status: complete saved: /mnt/d/JUNIOR/LabThayTho/GRACE-H3-CalibratedCascade/runs/experiment_summary.json
